In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

LOAD DATA

In [ ]:
folder_path = "/content/drive/MyDrive/Sensor_DL/MHEALTHDATASET"

dataframes = []

for file in os.listdir(folder_path):
    if file.endswith(".log"):
        df = pd.read_csv(os.path.join(folder_path, file), sep=r"\s+", header=None)
        df["subject"] = file
        dataframes.append(df)

dataset = pd.concat(dataframes, ignore_index=True)

# Assign columns
columns = [f"f{i}" for i in range(23)] + ["activity", "subject"]
dataset.columns = columns

 CLEAN

In [ ]:
dataset = dataset[dataset["activity"] != 0].reset_index(drop=True)

# Remove ECG (noise)
dataset.drop(columns=["f3", "f4"], inplace=True)

print("Cleaned:", dataset.shape)

Cleaned: (343195, 23)


SPLIT FIRST (IMPORTANT)

In [ ]:
train_df, test_df = train_test_split(
    dataset,
    test_size=0.2,
    random_state=42,
    stratify=dataset["activity"]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["activity"]
)

SEQUENCE CREATION

In [ ]:
TIME_STEPS = 50
STEP = 10

def create_sequences(df):
    Xs, ys = [], []

    for subject in df["subject"].unique():
        sub = df[df["subject"] == subject].reset_index(drop=True)

        X = sub.drop(columns=["activity", "subject"])
        y = sub["activity"]

        for i in range(0, len(X) - TIME_STEPS, STEP):
            Xs.append(X.iloc[i:i+TIME_STEPS].values)
            ys.append(y.iloc[i + TIME_STEPS - 1])

    return np.array(Xs), np.array(ys)

X_train, y_train = create_sequences(train_df)
X_val, y_val = create_sequences(val_df)
X_test, y_test = create_sequences(test_df)

print("Train:", X_train.shape)
print("Val:", X_val.shape)
print("Test:", X_test.shape)

Train: (24665, 50, 21)
Val: (2700, 50, 21)
Test: (6818, 50, 21)


SCALING (3D CORRECT)

In [ ]:
scaler = StandardScaler()

def scale(X, fit=False):
    shape = X.shape
    X = X.reshape(-1, shape[2])

    if fit:
        X = scaler.fit_transform(X)
    else:
        X = scaler.transform(X)

    return X.reshape(shape)

X_train = scale(X_train, fit=True)
X_val = scale(X_val)
X_test = scale(X_test)


In [ ]:
save_path = "/content/drive/MyDrive/Sensor_DL/final_data"
os.makedirs(save_path, exist_ok=True)

np.save(os.path.join(save_path, "X_train.npy"), X_train)
np.save(os.path.join(save_path, "y_train.npy"), y_train)

np.save(os.path.join(save_path, "X_val.npy"), X_val)
np.save(os.path.join(save_path, "y_val.npy"), y_val)

np.save(os.path.join(save_path, "X_test.npy"), X_test)
np.save(os.path.join(save_path, "y_test.npy"), y_test)

joblib.dump(scaler, os.path.join(save_path, "scaler.pkl"))

print("DATA READY (NO LEAKAGE)")

DATA READY (NO LEAKAGE)
